# ForgeLM Quick Start — Supervised Fine-Tuning (SFT)

Fine-tune a language model on your custom dataset using ForgeLM.

**Requirements:** GPU runtime recommended (Runtime → Change runtime type → T4 GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemililik/ForgeLM/blob/main/notebooks/quickstart_sft.ipynb)

In [ ]:
# Step 1: Install ForgeLM (with bitsandbytes for 4-bit quantization)
!pip install -q --no-cache-dir git+https://github.com/cemililik/ForgeLM.git bitsandbytes
!pip uninstall -y wandb -q  # Remove conflicting wandb (not needed)

# Verify installation
!forgelm --version

In [ ]:
# Step 2: Check GPU availability
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 1)
    print(f"GPU detected: {gpu_name} ({vram_gb} GB VRAM)")
else:
    print("WARNING: No GPU detected. Training will be very slow.")
    print("Go to Runtime → Change runtime type → T4 GPU")

In [ ]:
# Step 3: Create training config
import torch

use_4bit = torch.cuda.is_available()  # Enable 4-bit only with GPU

config_yaml = f"""
model:
  name_or_path: "HuggingFaceTB/SmolLM2-135M-Instruct"
  max_length: 512
  load_in_4bit: {str(use_4bit).lower()}
  backend: "transformers"

lora:
  r: 16
  alpha: 32
  dropout: 0.1
  target_modules:
    - "q_proj"
    - "v_proj"

training:
  output_dir: "./checkpoints"
  max_steps: 100              # Quick demo — ~2 min on T4. Remove for full training.
  per_device_train_batch_size: 4
  gradient_accumulation_steps: 2
  learning_rate: 2.0e-5
  eval_steps: 50
  save_steps: 50
  save_total_limit: 2

data:
  dataset_name_or_path: "timdettmers/openassistant-guanaco"
  shuffle: true
"""

with open("my_config.yaml", "w") as f:
    f.write(config_yaml)

print(f"Config saved! (model: SmolLM2-135M, 4-bit: {use_4bit}, max_steps: 100)")

In [ ]:
# Step 4: Validate config (dry run — no GPU needed)
!forgelm --config my_config.yaml --dry-run

In [ ]:
# Step 5: Start training!
!forgelm --config my_config.yaml

In [ ]:
# Step 6: Verify training output
import os

model_path = "./checkpoints/final_model"

if not os.path.exists(model_path):
    print(f"ERROR: Model directory '{model_path}' not found.")
    print("Training may have failed. Check the output above for errors.")
elif not os.path.isfile(os.path.join(model_path, "adapter_config.json")):
    print(f"ERROR: adapter_config.json not found in '{model_path}'.")
    print("Training may not have completed successfully.")
else:
    print("Training completed successfully!")
    print(f"\nModel saved to: {model_path}")
    print(f"Files: {os.listdir(model_path)}")

In [ ]:
# Step 7: Compare base model vs fine-tuned model
import os

from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

model_path = "./checkpoints/final_model"
base_model_name = "HuggingFaceTB/SmolLM2-135M-Instruct"

if not os.path.exists(model_path):
    print(f"Error: Model directory '{model_path}' not found.")
    print("Please ensure training completed successfully.")
elif not os.path.isfile(os.path.join(model_path, "adapter_config.json")):
    print(f"Error: adapter_config.json not found in '{model_path}'.")
else:
    # Load base model (before fine-tuning)
    print("Loading base model (before fine-tuning)...")
    base_model = AutoModelForCausalLM.from_pretrained(base_model_name)
    tokenizer = AutoTokenizer.from_pretrained(base_model_name)

    # Load fine-tuned model (after fine-tuning)
    print("Loading fine-tuned model (after fine-tuning)...")
    ft_model = PeftModel.from_pretrained(
        base_model.clone() if hasattr(base_model, "clone") else AutoModelForCausalLM.from_pretrained(base_model_name),
        model_path,
    )
    ft_tokenizer = AutoTokenizer.from_pretrained(model_path)

    # Test prompts — use questions that benefit from conversational fine-tuning
    test_prompts = [
        "What is machine learning?",
        "How do I learn Python?",
        "Explain the difference between AI and ML.",
    ]

    for prompt in test_prompts:
        messages = [{"role": "user", "content": prompt}]
        formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(formatted, return_tensors="pt")

        print(f"\n{'=' * 60}")
        print(f"PROMPT: {prompt}")
        print(f"{'=' * 60}")

        # Base model response
        base_outputs = base_model.generate(**inputs, max_new_tokens=150, do_sample=True, temperature=0.7)
        base_response = tokenizer.decode(base_outputs[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True)
        print(f"\n[BASE MODEL]:\n{base_response.strip()[:300]}")

        # Fine-tuned model response
        ft_inputs = ft_tokenizer(formatted, return_tensors="pt")
        ft_outputs = ft_model.generate(**ft_inputs, max_new_tokens=150, do_sample=True, temperature=0.7)
        ft_response = ft_tokenizer.decode(ft_outputs[0][ft_inputs["input_ids"].shape[1] :], skip_special_tokens=True)
        print(f"\n[FINE-TUNED]:\n{ft_response.strip()[:300]}")

    print(f"\n{'=' * 60}")
    print("Compare the responses above. Fine-tuned model should show")
    print("influence from the OpenAssistant conversational dataset.")